In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dill
import re

from tqdm.auto import tqdm
from spacy.matcher import PhraseMatcher
from spacy.util import filter_spans

In [ ]:
with open('../data/processed/hf_train_df.dill','rb') as f:
    hf_train_df=dill.load(f)
    
with open('../data/processed/hf_test_df.dill','rb') as f:
    hf_test_df=dill.load(f)

In [ ]:
def further_clean_text(text):
    # Remove special characters except spaces/./+/#
    text = re.sub(r'[^a-z0-9\s\+\#\.]', ' ', text)
        
    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

    #Remove extra whitespace — always LAST
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:
hf_train_df=hf_train_df.apply(lambda x: further_clean_text(x))
hf_test_df=hf_test_df.apply(lambda x: further_clean_text(x))

In [ ]:
import spacy
try:
    spacy.load("en_core_web_md")
except OSError:
    print("Downloading spaCy model 'en_core_web_md'...")
    !python -m spacy download en_core_web_md --quiet
    import spacy

In [ ]:
def build_phrase_matcher(nlp, alias_dict,skills):
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    patterns=[]
    skill_map={}
    # Add alias dict
    for key, values in alias_dict.items():
        patterns.append(nlp.make_doc(key))
        skill_map[key.lower()]=key
        for v in values:
            patterns.append(nlp.make_doc(v))
            skill_map[v.lower()]=key

     # 2. Add remaining skills (from Kaggle)
    for skill in skills:
        if skill.lower() not in skill_map:
            patterns.append(nlp.make_doc(skill))
            skill_map[skill.lower()] = skill


    matcher.add("SKILLS", patterns)
    return matcher,skill_map


def single_pass_pipeline(texts, matcher,skill_map, nlp):
    processed_texts = []
    print("Applying alias normalization and phrase matching...")


    print("Processing documents with spaCy...")
    for doc in tqdm(nlp.pipe(texts, batch_size=200,n_process=2), total=len(texts)):
        raw_matches = matcher(doc)
        spans = [(match_id, start, end) for match_id, start, end in raw_matches]
        filtered_spans = filter_spans([doc[start:end] for _, start, end in spans])

        span_dict = {}
        match_map = {(start, end): match_id for match_id, start, end in spans}

        for span in filtered_spans:
            match_id = match_map[(span.start, span.end)]
            normalize_skill=skill_map.get(span.text.lower(),span.text.lower())
            span_dict[span.start] = (span.end, normalize_skill)

        tokens = []
        i = 0

        while i < len(doc):
            if i in span_dict:
                end_idx, label = span_dict[i]
                tokens.append(label.replace(" ","_"))
                i = end_idx
                continue

            token = doc[i]

            if (token.is_stop or token.is_punct or token.is_space or
                token.like_url or token.like_email):
                i += 1
                continue

            if token.pos_ not in ['NOUN', 'VERB', 'ADJ', 'PROPN']:
                i += 1
                continue

            lemma = token.lemma_.lower()

            if len(lemma) > 2 :
                tokens.append(lemma)

            i += 1

        processed_texts.append(" ".join(tokens))

    return processed_texts


def preprocess_pipeline(texts, alias_dict,skills):
    nlp = spacy.load("en_core_web_md",disable=["parser", "ner"])
    print("Building phrase matcher...")
    matcher,skill_map =build_phrase_matcher(nlp, alias_dict,skills)
    return single_pass_pipeline(texts,matcher,skill_map,nlp)


In [ ]:
resume_processed_train = preprocess_pipeline(hf_train_df['resume_text'].tolist(), tech_aliases,skills)
job_description_processed_train = preprocess_pipeline(hf_train_df['job_description_text'].tolist(), tech_aliases,skills)


resume_processed_test = preprocess_pipeline(hf_test_df['resume_text'].tolist(), tech_aliases,skills)
job_description_processed_test = preprocess_pipeline(hf_test_df['job_description_text'].tolist(), tech_aliases,skills)


In [ ]:
hf_train_df['resume_clean']=resume_processed_train
hf_train_df['jd_clean']=job_description_processed_train

hf_test_df['resume_clean']=resume_processed_test
hf_test_df['jd_clean']=job_description_processed_test

In [ ]:
hf_train_df.head()

In [ ]:
label_map = {
    "No Fit": 0,
    "Potential Fit": 1,
    "Good Fit": 2
}

hf_train_df['label'] = hf_train_df['label'].map(label_map)
hf_test_df['label'] = hf_test_df['label'].map(label_map)